In [1]:
!pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [2]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os

print("All libaries imported")

All libaries imported


In [ ]:
GROQ_API_KEY = ""
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API client initialized")

Groq API client initialized


In [4]:
df = pd.read_csv("college_notes.csv")
print("First 5 lines of the dataset :\n")
df.head(5)



First 5 lines of the dataset :



,note_id,subject,topic,content
0,1,Data Engineering,ETL Pipelines,"ETL stands for Extract, Transform, Load. It is..."
1,2,Data Engineering,Data Warehousing,A data warehouse is a central repository that ...
2,3,Data Engineering,Apache Spark,Apache Spark is an open-source distributed com...
3,4,Data Engineering,Medallion Architecture,Medallion Architecture is a data design patter...
4,5,Data Engineering,Data Pipelines,A data pipeline is a series of automated steps...


In [5]:
print("Last  5 lines of the dataset :\n")
df.tail(5)

Last  5 lines of the dataset :



,note_id,subject,topic,content
10,11,GenAI,Embeddings,An embedding is a numerical representation of ...
11,12,GenAI,Vector Databases,A vector database is a specialized database sy...
12,13,GenAI,RAG Systems,Retrieval-Augmented Generation or RAG is an AI...
13,14,Python,Pandas Library,Pandas is a Python library for data manipulati...
14,15,Python,API Integration,An API or Application Programming Interface al...


In [6]:
print("subjects in the Dataset:")
print(df['subject'].value_counts())

print("\nSample of Topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of Content (number of characters) for each note:")
df['content_length']=df['content'].apply(len)
print(df[['topic', 'content_length']].to_string(index=False))

subjects in the Dataset:
subject
Data Engineering    5
GenAI               5
Machine Learning    3
Python              2
Name: count, dtype: int64

Sample of Topics:
 note_id          subject                  topic
       1 Data Engineering          ETL Pipelines
       2 Data Engineering       Data Warehousing
       3 Data Engineering           Apache Spark
       4 Data Engineering Medallion Architecture
       5 Data Engineering         Data Pipelines
       6 Machine Learning      Linear Regression
       7 Machine Learning    Feature Engineering
       8 Machine Learning       Model Evaluation
       9            GenAI  Large Language Models
      10            GenAI     Prompt Engineering
      11            GenAI             Embeddings
      12            GenAI       Vector Databases
      13            GenAI            RAG Systems
      14           Python         Pandas Library
      15           Python        API Integration

Length of Content (number of characters) for each

In [7]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}"for row in df.to_dict('records')]
metadatas = [
    {"subject":row['subject'],"topic":row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared:{len(documents)}")
print(f"First document ID :{ids[0]}")
print(f"First metadata  : {metadatas[0]}")
print(f"First 100 chars of doc:{documents[0][:100]}...")

Total chunks prepared:15
First document ID :note_1
First metadata  : {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc:ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from ...


In [8]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("\n Embedding model loaded successfully.")
test_embedding = embedding_model.encode("This is a test sentence.")
print(f"Test embedding shape:{test_embedding.shape}")
print(f"First 5 value of test embedding:{test_embedding[:5]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


 Embedding model loaded successfully.
Test embedding shape:(384,)
First 5 value of test embedding:[0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [9]:
chroma_client=chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag")
print("ChromaDB client created")
print(f"Collection name: college_notes_rag")
print(f"Documents in collection so far:{collection.count()}")

ChromaDB client created
Collection name: college_notes_rag
Documents in collection so far:0


In [10]:
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(embeddings.shape)
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)
print(collection.count())

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(15, 384)


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 33.7MiB/s]


15


In [11]:
collection.add(
    documents = documents,
    embeddings = embeddings,
    ids=ids,
    metadatas=metadatas
)
print(f"\nDocuments successfully added")
print(f"Total document in collection:{collection.count()}")


Documents successfully added
Total document in collection:15


In [12]:
def retrieve_relevant_chunks(question,top_k=3):
  question_embedding=embedding_model.encode(question)
  results=collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )
  return results

In [13]:
def retrieve_relevant_chunks(question, top_k = 3):
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings = [question_embedding],
      n_results=top_k
  )
  return results

print("retrieval ")

retrieval 


In [14]:
test_question = "What is ETL and how does it work "
print(f"Test question :{test_question}")
print("="*60)

results = retrieve_relevant_chunks(test_question,top_k=3)
print("\nTop 3 Retireved Chunks:")
print("="*60)

for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
  print(f"Result{i+1}:")
  print(f"subject:{meta['subject']}")
  print(f"Topic:{meta['topic']}")
  print(f"Distance:{dist:.4f}")
  print(f"Content:{doc[:120]}...")

Test question :What is ETL and how does it work 

Top 3 Retireved Chunks:
Result1:
subject:Data Engineering
Topic:ETL Pipelines
Distance:0.4291
Content:ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from source systems into ...
Result2:
subject:Python
Topic:API Integration
Distance:1.5107
Content:An API or Application Programming Interface allows different software systems to communicate with each other. In Python,...
Result3:
subject:Data Engineering
Topic:Medallion Architecture
Distance:1.5478
Content:Medallion Architecture is a data design pattern used in modern data lakes and lakehouses. It organizes data into three l...


In [15]:
def build_context_from_results(results):
  context_parts=[]
  for i,(doc,meta)in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):

    chunk_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}]\n{doc}"
    context_parts.append(chunk_text)
  context = "\n\n--\n\n".join(context_parts)
  return context

test_question = "What is ETL and how does it work "
results = retrieve_relevant_chunks(test_question,top_k=3)
context = build_context_from_results(results)
print(context)

[Source 1: Data Engineering - ETL Pipelines]
ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from source systems into a data warehouse. First, data is extracted from multiple sources such as databases, APIs, or files. Then it is transformed by cleaning errors, converting formats, and applying business rules. Finally, the cleaned data is loaded into a destination system such as a data warehouse or data lake. ETL pipelines are automated to run on a schedule, ensuring fresh data is always available for analysis.

--

[Source 2: Python - API Integration]
An API or Application Programming Interface allows different software systems to communicate with each other. In Python, the requests library is used to call REST APIs. A GET request retrieves data from a server. A POST request sends data to a server. APIs return responses in JSON format which can be parsed into Python dictionaries. Authentication is handled using API keys passed in request he

In [21]:
def generate_rag_answer(question,context):
  system_prompt = """
    You are a helpful academic assistant for engineering students.
    You will be given context retrieved from a college knowledge base, and a student's question.
    RULES:
    1. Answer ONLY using the informatiion provided in the context below.
    2. If the answer is not found in the context, say exactly:
       "I don't have enough information in my knowledge base to answer this question."
    3. Do not use your general training knowledge.
    4. Keep answers clear, accurate, and beginner-friendly.
    5. Mention which source the information came from when possible."""


  user_prompt = f"""Context from Knowledge Base:{context}
  ---
  Student's Question: {question}
  Please answer the question based only on the context provided above."""

  response=groq_client.chat.completions.create(
      model='llama-3.1-8b-instant',
      messages=[
          {"role":"system","content":system_prompt},
          {"role":"user","content":user_prompt}
      ],
      temperature=0.1,
      max_tokens=500
  )
  answer=response.choices[0].message.content
  return answer

In [19]:
def ask_college_assistant(question,top_k=3,verbose=True):
  if verbose:
    print(f"Question: {question}")
  results=retrieve_relevant_chunks(question,top_k=top_k)
  if verbose:
    print(f"Retrieved {top_k}")
    for i,meta in enumerate(results['metadatas'][0]):
      print(f" {i+1},{meta['subject']}-{meta['topic']}")
    print("\n Step2:Building content string...")
  context=build_context_from_results(results)
  if verbose:
    print(f"Context built({len(context)}) characters")
    print(f"\n Step3:Sending to LLM for answer generation")

  answer=generate_rag_answer(question,context)
  if verbose:
    print(f"Answer: {answer}")
  return answer

The `AttributeError` has been fixed in `generate_rag_answer`. Let's re-run the `ask_college_assistant` function to confirm.

In [22]:
question_1 = "What is ETL and What are its three main stages?"

answer_1 = ask_college_assistant(question_1, top_k=3, verbose=True)

Question: What is ETL and What are its three main stages?
Retrieved 3
 1,Data Engineering-ETL Pipelines
 2,GenAI-Prompt Engineering
 3,Data Engineering-Medallion Architecture

 Step2:Building content string...
Context built(1588) characters

 Step3:Sending to LLM for answer generation
Answer: Based on the context provided, I can answer the student's question.

ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from source systems into a data warehouse.

The three main stages of ETL are:

1. **Extract**: Data is extracted from multiple sources such as databases, APIs, or files.
2. **Transform**: Data is transformed by cleaning errors, converting formats, and applying business rules.
3. **Load**: The cleaned data is loaded into a destination system such as a data warehouse or data lake.

This information comes from [Source 1: Data Engineering - ETL Pipelines].
